# Study 953 — Replicating the Convert 🎭

**Is a convertible-bond fund anything more than equity plus credit in a costume?**

A convertible bond is sold as an instrument you cannot build from parts: a bond that turns
into a share, so you ride the stock up while a bond floor catches you on the way down — a
**convex** payoff worth paying a fee for. That is a testable claim, and the test is
mechanical. Fit the fund to a plain, static, long-only mix of liquid ETFs; **freeze** the
recipe; then see (a) whether anything is left over, and (b) whether what is left over has
the promised *shape*.

We test **CWB** (the $4bn SPDR convertibles ETF) against **SPY + QQQ + LQD + cash (BIL)**,
daily **total-return** closes, 2009-04-16 → 2026-06-30 (4,328 days), every leg
**excess-of-cash**. The mix is fitted on **2009-2017 only** and scored on the untouched
**2018-01-02 → 2026-06-30** hold-out.

*Numbers below are the frozen headline (`docs/results.md`, Fingerprint `5e5674563124`); the only
live cells run the fast offline synthetic control. As-of 2026-06-30.*


## 1. Three ETFs write the recipe

We did not choose the weights — the data did, using only the fund's first nine years. The answer is startling in its plainness.

In [1]:
R = {'start': '2009-04-16', 'end': '2026-06-30', 'n_days': 4328, 'fp': '5e5674563124', 'is_end': '2017-12-31', 'oos_start': '2018-01-02', 'w_spy': 36.7, 'w_qqq': 20.2, 'w_lqd': 5.8, 'w_cash': 37.3, 'r2_is': 0.703, 'te_is': 5.68, 'fit_intercept': 1.12, 'alpha_is': 1.28, 't_is': 0.8, 'n_is': 2193, 'alpha_oos': 1.8, 't_oos': 0.58, 'te_oos': 8.3, 'corr_oos': 0.85, 'n_oos': 2134, 'ci_lo': -4.47, 'ci_hi': 7.55, 'ci_pos': 71.4, 'sharpe_fund': 0.662, 'sharpe_repl': 0.725, 'sharpe_gap': -0.063, 'vol_fund': 15.51, 'vol_repl': 11.69, 'dd_fund': -32.24, 'dd_repl': -19.46, 'dd_repl_volmatched': -25.25, 'm_r2': 0.764, 'm_te': 7.85, 'n_months': 102, 'ter_low': -55.8, 'ter_mid': 29.9, 'ter_high': 72.5, 't_low': -1.57, 't_high': 1.56, 'smile': -21.6, 't_smile': -0.5, 'up_cap': 1.219, 'dn_cap': 1.215, 'asym': 0.004, 'tm_gamma': 0.192, 't_tm': 0.25, 'era_e_alpha': 4.68, 'era_e_t': 0.86, 'era_e_n': 1008, 'era_l_alpha': -0.78, 'era_l_t': -0.23, 'era_l_n': 1126, 'refit_alpha': 1.35, 'refit_t': 0.61, 'refit_n': 3141, 'cost0_alpha': 1.69, 'cost0_t': 0.54, 'cost10_alpha': 1.91, 'cost10_t': 0.62, 'kit_spy_alpha': 3.01, 'kit_spy_t': 0.92, 'kit_spylqd_alpha': 2.99, 'kit_spylqd_t': 0.94, 'kit_spyqqq_alpha': 1.81, 'kit_spyqqq_t': 0.57, 'n_specs': 11, 'max_spec_t': 0.94, 'split_best_alpha': 2.35, 'split_best_t': 0.9, 'split_worst_alpha': -2.72, 'split_worst_t': -0.8, 'icvt_alpha': -2.16, 'icvt_t': -0.56, 'icvt_up': 0.802, 'icvt_dn': 0.871, 'icvt_n': 1378, 'mar20_fund': -13.28, 'mar20_repl': -6.49, 'mar20_gap': -6.79, 'fund_fee': 0.4, 'replica_fee_bps': 10, 'syn_alpha': 5.02, 'syn_t': 3.97, 'syn_gamma': 0.73, 'syn_t_gamma': 3.45, 'syn_smile': 26.7, 'syn_null_mean': 0.03, 'syn_null_sd': 0.87, 'syn_null_fire': 0}
print('What eight years of CWB returns say it actually is:')
for leg, w in [('SPY  (US large cap)', R['w_spy']), ('QQQ  (Nasdaq growth)', R['w_qqq']),
               ('LQD  (IG credit)', R['w_lqd']), ('cash (T-bills)', R['w_cash'])]:
    print(f'  {leg:22s} {w:5.1f}%')
print(f"\nin-sample fit quality: R2 {R['r2_is']:.3f}")

What eight years of CWB returns say it actually is:
  SPY  (US large cap)     36.7%
  QQQ  (Nasdaq growth)    20.2%
  LQD  (IG credit)         5.8%
  cash (T-bills)          37.3%

in-sample fit quality: R2 0.703


The 'bond' half of a convertible is **not** a bond. Fitted freely, the fund wants **37% cash** and barely **6% investment-grade credit**. Read plainly: a convertible fund is roughly *half an equity fund and a third a money-market fund*, with a sliver of credit — and the equity half leans Nasdaq, because the companies that issue converts are growth companies that cannot borrow cheaply.

> 🔬 **For the quants** — the weights come from Sharpe-style constrained least squares (non-negative, summing to at most one), so the replica is a portfolio someone could actually hold. The constraint never binds: the unconstrained OLS solution is already inside it, so no leg goes short and no borrow cost arises.

## 2. Freeze it, and see what is left

The honest test is not how well a recipe fits the past — it is what the recipe misses on years it has never seen. So we lock the weights at the end of 2017 and never touch them again.

In [2]:
print(f"hold-out {R['oos_start']} -> {R['end']}  ({R['n_oos']:,} trading days)")
print(f"  what the fund did that the frozen mix did not: {R['alpha_oos']:+.2f}%/yr")
print(f"  is that distinguishable from zero?  HAC t = {R['t_oos']:+.2f}   "
      f"(the desk's bar is |t| >= 2)")
print(f"  95% confidence range: [{R['ci_lo']:+.2f}%, {R['ci_hi']:+.2f}%]  -> straddles zero")
print(f"\nrisk-adjusted return (excess of cash):")
print(f"  the fund     {R['sharpe_fund']:+.3f}")
print(f"  the frozen mix {R['sharpe_repl']:+.3f}   <- the copy edged the original")

hold-out 2018-01-02 -> 2026-06-30  (2,134 trading days)
  what the fund did that the frozen mix did not: +1.80%/yr
  is that distinguishable from zero?  HAC t = +0.58   (the desk's bar is |t| >= 2)
  95% confidence range: [-4.47%, +7.55%]  -> straddles zero

risk-adjusted return (excess of cash):
  the fund     +0.662
  the frozen mix +0.725   <- the copy edged the original


A nine-year-old recipe, never updated, kept pace with the fund it was copied from — and on risk-adjusted terms very slightly beat it. Nothing here is *wrong* with CWB; the point is that nothing here is *distinctive* about it either.

## 3. The claim that actually matters: the shape

Convertibles are not sold on return, they are sold on **shape** — win more in good months, lose less in bad ones. Measure it: sort the hold-out months by how the stock market did, and see how the fund fared against its own copy in each third.

In [3]:
print('fund minus its frozen copy, by third of the stock market (bps per month):')
print(f"  worst months   {R['ter_low']:+7.1f}   <- should be POSITIVE if a bond floor exists")
print(f"  middle months  {R['ter_mid']:+7.1f}")
print(f"  best months    {R['ter_high']:+7.1f}   <- positive, as promised")
print(f"\nthe promised smile (both tails better than the middle): {R['smile']:+.1f} bps/month")
print(f"upside captured vs the copy   {R['up_cap']:.3f}")
print(f"downside captured vs the copy {R['dn_cap']:.3f}")
print(f"asymmetry (the whole pitch)   {R['asym']:+.3f}   <- essentially zero")

fund minus its frozen copy, by third of the stock market (bps per month):
  worst months     -55.8   <- should be POSITIVE if a bond floor exists
  middle months    +29.9
  best months      +72.5   <- positive, as promised

the promised smile (both tails better than the middle): -21.6 bps/month
upside captured vs the copy   1.219
downside captured vs the copy 1.215
asymmetry (the whole pitch)   +0.004   <- essentially zero


There is the answer, and it is not subtle. CWB does beat its copy in strong months — by capturing **22% more** of the move. It also loses **22% more** in weak ones. That is not convexity; that is **leverage**. You could buy the same thing by holding a bit more of the copy.

> 🔬 **For the quants** — the Treynor-Mazuy curvature on the fund-minus-replica difference is γ = **+0.19** with HAC *t* = **+0.25**: indistinguishable from a straight line. The tail 'smile' is in fact **-21.6** bps/month (Welch *t* = -0.50) — the tails are *worse* than the middle, the opposite of a smile.

## 4. The one month the floor was asked for

March 2020 is the cleanest test a bond floor will ever get.

In [4]:
print(f"March 2020, excess of cash:")
print(f"  CWB (the convertible fund)  {R['mar20_fund']:+.2f}%")
print(f"  its own frozen stock/credit/cash copy {R['mar20_repl']:+.2f}%")
print(f"  shortfall {R['mar20_gap']:+.2f}%  in a single month")

March 2020, excess of cash:
  CWB (the convertible fund)  -13.28%
  its own frozen stock/credit/cash copy -6.49%
  shortfall -6.79%  in a single month


The convertible fund lost roughly **twice** what its own plain replica lost, in the month the floor was supposed to matter. When arbitrageurs are forced to delever, convertibles trade *below* the value of their parts — the floor gives way exactly when you reach for it.

## 5. Is the measuring stick honest? (a live offline check)

Before believing a blank result, prove the instrument works. We build a fake fund that genuinely *is* a static mix plus a real 2%/yr edge and a real ratcheting equity exposure — the honest version of the convertible story — and check the same machinery finds it. Then we switch both effects off and check it finds nothing.

In [5]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from convert_repl import data, strategy as st
planted = st.synthetic_detect(data.synthetic_panel(signal_strength=1.0, seed=953)[0])
print('planted fund (there IS something to find):')
print('  leftover return %+.2f%%/yr (t %+.2f)  |  curvature %+.2f (t %+.2f)'
      % (planted['alpha_oos_ann']*100, planted['t_oos'],
         planted['tm_gamma'], planted['t_tm_gamma']))
nulls = [st.synthetic_detect(data.synthetic_panel(signal_strength=0.0, seed=953+s)[0])
         for s in range(4)]
print('null funds, 4 fresh draws (there is NOTHING to find):')
for i, n in enumerate(nulls):
    print('  draw %d: leftover %+.2f%%/yr (t %+.2f)  |  curvature %+.2f (t %+.2f)'
          % (i + 1, n['alpha_oos_ann']*100, n['t_oos'], n['tm_gamma'], n['t_tm_gamma']))
print('  -> not one of them clears the |t| >= 2 bar')

planted fund (there IS something to find):
  leftover return +5.02%/yr (t +3.97)  |  curvature +0.73 (t +3.45)


null funds, 4 fresh draws (there is NOTHING to find):
  draw 1: leftover +1.37%/yr (t +1.33)  |  curvature -0.18 (t -1.21)
  draw 2: leftover -0.32%/yr (t -0.30)  |  curvature +0.05 (t +0.55)
  draw 3: leftover +0.30%/yr (t +0.28)  |  curvature +0.06 (t +0.38)
  draw 4: leftover -1.64%/yr (t -1.57)  |  curvature -0.06 (t -0.39)
  -> not one of them clears the |t| >= 2 bar


The instrument rings loudly when there is something there and stays quiet when there is not. So the blank we got on the real tape is a fact about convertible ETFs, not a broken ruler. *(This cell is synthetic — a machinery proof, never market evidence.)*

## Verdict

- **Signal — None.** Out-of-sample, the fund's leftover return over a frozen SPY/QQQ/LQD/cash mix is **+1.80%/yr with *t* = +0.58** — zero by any standard — and it flips sign between eras (+4.68%/yr in 2018-21, -0.78%/yr since 2022). The second convertible ETF (ICVT) gives -2.16%/yr (*t* = -0.56). The convexity claim fails on its own terms: up-capture 1.219 versus down-capture 1.215.
- **Tradability — Mirage.** There is nothing to trade in either direction. Owning the fund buys a coin-flip residual against a certain 0.40%/yr fee; betting *against* the fund means running a 7.8%/yr tracking error to harvest a 2%/yr expectation, with the bad months bunched (−6.7% in March 2020 alone). The useful conclusion is not a trade — it is that you already own this exposure.